# Taste-Bench replay: grading the middle of an agent's run

Executable walkthrough of the paper's core numbers (Microsoft + City University of Hong Kong, Oct 18 2026):
the 25% guessing floor, the 62.3→21.0% time-horizon collapse, and why distilled judgment captures ~78% of the possible gain.
All numbers are the paper's reported figures; the code below recomputes the derived claims.

In [ ]:
import numpy as np
rng = np.random.default_rng(7)

# 1. Guessing floor: credit only if right on BOTH answer orders
p_both = 0.5 * 0.5
print(f"coin-flip accuracy under double-order grading: {p_both:.1%}")
assert abs(p_both - 0.25) < 1e-9, "guessing floor must be 25%"

## 2. Time-horizon collapse
Paper's headline curve: mean 14-model accuracy falls from 62.3% (clue visible in the past) to 21.0% (clue only far ahead).

In [ ]:
# reported accuracies across the four evidence-horizon tiers
tiers = np.array(["clue in past", "tier 2", "tier 3", "clue far ahead"])
acc  = np.array([62.3, 55.0, 38.0, 21.0])  # middle tiers interpolated from the published curve
print(f"accuracy drop: {acc[0]:.1f}% -> {acc[-1]:.1f}%  (delta {acc[0]-acc[-1]:.1f} pts)")
print(f"late-evidence accuracy {acc[-1]:.1f}% is BELOW the 25% coin-flip floor: {acc[-1] < 25.0}")
assert acc[0] == 62.3 and acc[-1] == 21.0
assert acc[-1] < 0.25*100  # worse than guessing when evidence arrives late

## 3. Decision-fork simulator: hindsight as a free grader
Model one fork: two paths start identically at the fork, then accumulate noisy per-step rewards.
The *true* better path is only revealed by the full trajectory (hindsight); a judge that sees
only the first k steps must guess from partial evidence.

In [ ]:
def simulate_forks(n_forks=2000, steps=40, gap=0.35, late_from=30, seed=7):
    """Two paths per fork; both are noisy random walks, but path 0 gets a
    reward edge `gap` per step only AFTER step `late_from` — the deciding
    evidence arrives LATE, like the hardest Taste-Bench tier.
    The hindsight label (what the paper's generator uses) is simply which
    path ended up better once the full trajectory is known."""
    r = np.random.default_rng(seed)
    walk = r.normal(0.0, 1.0, (n_forks, 2, steps)).cumsum(axis=2)
    walk[:, 0, late_from:] += gap * np.arange(1, steps - late_from + 1)
    labels = (walk[:, 0, -1] < walk[:, 1, -1]).astype(int)  # 0 -> path A better
    return labels, walk[:, 0, :], walk[:, 1, :]

def judge(labels, path_a, path_b, k_steps):
    """Judge sees only the first k steps and picks the path with the higher partial total."""
    pick = np.stack([path_a[:, k_steps-1], path_b[:, k_steps-1]], axis=1).argmax(axis=1)
    return (pick == labels).mean()

labels, pa, pb = simulate_forks()
hindsight_acc = judge(labels, pa, pb, k_steps=40)   # full trajectory = the label itself
early_acc     = judge(labels, pa, pb, k_steps=4)    # only pre-fork-ish steps visible
print(f"judge with full hindsight: {hindsight_acc:.3f}  (this IS the label)")
print(f"judge with 4-step evidence: {early_acc:.3f}  (~near coin flip {0.5:.1f})")
assert hindsight_acc == 1.0
assert abs(early_acc - 0.5) < 0.10  # late-arriving evidence -> early judge is guessing


## 4. Distillation payoff: how much of the possible gain did the student capture?
Paper figures: judgment 30.0% → 47.9%; SWE-bench Pro success 14.6% → 33.7%; perfect advice ceiling 39.0%.

In [ ]:
judge_before, judge_after = 30.0, 47.9
swe_before, swe_student, swe_ceiling = 14.6, 33.7, 39.0
print(f"judgment gain: +{judge_after - judge_before:.1f} pts on unseen tasks")
print(f"SWE-bench Pro success: {swe_before}% -> {swe_student}% (x{swe_student/swe_before:.2f})")
captured = (swe_student - swe_before) / (swe_ceiling - swe_before)
print(f"fraction of possible gain captured by the student: {captured:.1%}")
assert abs(captured - 0.783) < 0.005  # ~78% of the possible gain

# scale sanity: the 2x success claim
assert swe_student / swe_before > 2.0, "student advice more than doubles agent success"